# MAINTAIN AI V1.5 — Pretrained Model Audit (NO TRAINING)

This notebook uses **released pretrained weights** first. It does not train LSTM/Transformer/TCN models.

Primary zero-shot candidate: **TimeRadar**, which ships a ready-to-use Hugging Face-format checkpoint and accepts `[batch, sequence_length, channels]`; its released checkpoint requires sequence length 100 while channel count may vary. urlTimeRadarhttps://github.com/mala-lab/TimeRadar

Secondary zero-shot candidate: **Chronos-2**, a pretrained forecasting foundation model supporting univariate and multivariate forecasting. It can provide a temporal forecasting signal, but it is not itself a failure-risk classifier. urlChronos-2https://github.com/amazon-science/chronos-forecasting

**Important correction:** ChronosAD is an architecture that uses a pretrained Chronos backbone plus a custom BiLSTM/attention block; its official README instructs users to train/test that pipeline. Therefore it is not treated here as a completely pretrained end-to-end anomaly detector. urlChronosADhttps://github.com/intelligolabs/ChronosAD

In [ ]:
# Colab's current Python 3.13 cannot install TimeRadar's pinned `transformers==4.40.1` because its old `tokenizers` dependency has no compatible wheel here. Do not downgrade the runtime.
# Use a modern Transformers release plus the actual TimeRadar runtime dependency. We also deliberately DO NOT reinstall torch, because Colab already supplies a CUDA-enabled PyTorch build.
!pip -q install -U "transformers>=4.45,<5" "torch-frft>=0.8.1" "safetensors>=0.7.0"
import os,sys,json,subprocess
from pathlib import Path
import numpy as np,pandas as pd,torch
ROOT=Path('/content/maintain_ai_v1_3'); OUT=ROOT/'artifacts_pretrained_v1_5'; OUT.mkdir(parents=True,exist_ok=True)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:',DEVICE); print('python:',sys.version); print('torch:',torch.__version__)
import transformers, torch_frft
print('transformers:',transformers.__version__,'torch_frft:',getattr(torch_frft,'__version__','installed'))

## 1. Prepare MAINTAIN AI MetroPT-3 data automatically

**This notebook is self-contained.** Colab runtimes are temporary, so it must not assume that a previous notebook created `/content/maintain_ai_v1_3/temporal_sequences_v1_3`. If the sequence files already exist, they are reused. Otherwise this section downloads the official MetroPT-3 archive and builds the 1-minute / 24-step evaluation set.

In [ ]:
SEQ=ROOT/'temporal_sequences_v1_3'; SEQ.mkdir(parents=True,exist_ok=True)
if not (SEQ/'metro_X.npy').exists():
    import zipfile, requests
    from io import BytesIO
    url='https://archive.ics.uci.edu/static/public/791/metropt%2B3%2Bdataset.zip'
    raw=ROOT/'raw'/'industrial'/'metropt3'; raw.mkdir(parents=True,exist_ok=True)
    zip_path=raw/'metropt3.zip'
    if not zip_path.exists():
        print('Downloading MetroPT-3 (~218 MB)...')
        with requests.get(url,stream=True,timeout=300) as r:
            r.raise_for_status()
            with open(zip_path,'wb') as f:
                for chunk in r.iter_content(8*1024*1024):
                    if chunk: f.write(chunk)
    csv_path=raw/'MetroPT3(AirCompressor).csv'
    if not csv_path.exists():
        with zipfile.ZipFile(zip_path) as z:
            names=z.namelist(); print('archive files:',names)
            target=[n for n in names if n.lower().endswith('.csv') and 'metropt3' in n.lower()][0]
            with z.open(target) as src, open(csv_path,'wb') as dst: dst.write(src.read())
    df=pd.read_csv(csv_path)
    df['timestamp']=pd.to_datetime(df['timestamp'],utc=True)
    signal_cols=['TP2','TP3','H1','DV_pressure','Reservoirs','Oil_temperature','Motor_current','COMP','DV_eletric','Towers','MPG','LPS','Pressure_switch','Oil_level','Caudal_impulses']
    df=df[['timestamp']+signal_cols].sort_values('timestamp')
    for c in signal_cols: df[c]=pd.to_numeric(df[c],errors='coerce')
    minute=df.set_index('timestamp')[signal_cols].resample('1min').agg(['mean','std','min','max'])
    minute.columns=[f'{a}_{b}' for a,b in minute.columns]
    minute=minute.dropna(how='all').ffill(limit=2).bfill(limit=2)
    # UCI/NASA-style documented MetroPT-3 failure intervals.
    events=[pd.Timestamp('2020-04-18 00:00:00',tz='UTC'),pd.Timestamp('2020-05-29 23:30:00',tz='UTC'),pd.Timestamp('2020-06-05 10:00:00',tz='UTC'),pd.Timestamp('2020-07-15 14:30:00',tz='UTC')]
    active=[]
    active_ranges=[(pd.Timestamp('2020-04-18 00:00:00',tz='UTC'),pd.Timestamp('2020-04-19 00:00:00',tz='UTC')),(pd.Timestamp('2020-05-29 23:30:00',tz='UTC'),pd.Timestamp('2020-05-30 06:00:00',tz='UTC')),(pd.Timestamp('2020-06-05 10:00:00',tz='UTC'),pd.Timestamp('2020-06-07 14:30:00',tz='UTC')),(pd.Timestamp('2020-07-15 14:30:00',tz='UTC'),pd.Timestamp('2020-07-15 19:00:00',tz='UTC'))]
    idx=minute.index
    # Generic future-event targets: next failure start strictly after prediction time.
    e=np.array([x.value/1e9 for x in events]); tsec=idx.view('int64')/1e9
    future=np.full((len(idx),3),np.nan,dtype=np.float32)
    active_mask=np.zeros(len(idx),dtype=bool)
    for a,b in active_ranges: active_mask|=(idx>=a)&(idx<b)
    for i,ts in enumerate(tsec):
        j=np.searchsorted(e,ts,side='right')
        if j<len(e):
            delta=e[j]-ts
            for k,h in enumerate([24*3600,48*3600,7*24*3600]): future[i,k]=float(delta<=h and not active_mask[i])
    # Event-aware evaluation windows. Validation targets Event 3; test targets Event 4.
    def override_target(start,end,event_start):
        m=(idx>=start)&(idx<end)&(~active_mask)
        d=(event_start.value/1e9)-tsec
        future[m,0]=(d[m]>0)&(d[m]<=24*3600); future[m,1]=(d[m]>0)&(d[m]<=48*3600); future[m,2]=(d[m]>0)&(d[m]<=7*24*3600)
    override_target(pd.Timestamp('2020-05-29 10:00:00',tz='UTC'),pd.Timestamp('2020-06-05 10:00:00',tz='UTC'),events[2])
    override_target(pd.Timestamp('2020-07-08 14:30:00',tz='UTC'),pd.Timestamp('2020-07-15 14:30:00',tz='UTC'),events[3])
    # Build only contiguous 24-minute histories so large telemetry gaps cannot become fake sequences.
    vals=minute.to_numpy(dtype=np.float32); times=idx.to_numpy(); rows=[]; ys=[]; ts_out=[]
    for end in range(23,len(minute)):
        if (idx[end]-idx[end-23]).total_seconds()!=23*60: continue
        y=future[end]
        if np.isnan(y).any(): continue
        rows.append(vals[end-23:end+1]); ys.append(y); ts_out.append(times[end])
    X=np.asarray(rows,dtype=np.float32); Y=np.asarray(ys,dtype=np.float32); T=np.asarray(ts_out)
    np.save(SEQ/'metro_X.npy',X); np.save(SEQ/'metro_y.npy',Y); np.save(SEQ/'metro_times.npy',T)
    ts=pd.to_datetime(T,utc=True)
    test_start=pd.Timestamp('2020-07-08 14:30:00',tz='UTC'); test_end=pd.Timestamp('2020-07-15 14:30:00',tz='UTC')
    val_start=pd.Timestamp('2020-05-29 10:00:00',tz='UTC'); val_end=pd.Timestamp('2020-06-05 10:00:00',tz='UTC')
    test=np.flatnonzero((ts>=test_start)&(ts<test_end)); val=np.flatnonzero((ts>=val_start)&(ts<val_end)); train=np.setdiff1d(np.arange(len(ts)),np.union1d(val,test))
    np.savez(SEQ/'metro_splits.npz',train=train,val=val,test=test)
    print('Built:',X.shape,'targets:',Y.shape,'train/val/test:',len(train),len(val),len(test))
else: print('Existing Metro sequence files found; reusing them.')

X=np.load(SEQ/'metro_X.npy',mmap_mode='r'); Y=np.load(SEQ/'metro_y.npy',mmap_mode='r').astype(np.float32); T=np.load(SEQ/'metro_times.npy',allow_pickle=True)
spl=np.load(SEQ/'metro_splits.npz'); tr,va,te=[spl[k] for k in ('train','val','test')]
print('sequence data:',X.shape,'targets:',Y.shape,'test:',len(te)); print('test future-risk rates:',Y[te].mean(0))

## 2. Clone TimeRadar source only

The previous notebook attempted `pip install git+...`. That failed because the TimeRadar repository is an application/research repository, not a pip package. We only clone it and load the included pretrained checkpoint.

In [ ]:
TR=OUT/'TimeRadar'
if not TR.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/mala-lab/TimeRadar.git',str(TR)],check=True)
MODEL_DIR=TR/'TimeRadar'
print('checkpoint directory:',MODEL_DIR)
print('exists:',MODEL_DIR.exists())
print('files:',[str(p.relative_to(MODEL_DIR)) for p in MODEL_DIR.rglob('*') if p.is_file()][:40])

## 3. Load the actual pretrained TimeRadar checkpoint

No fitting occurs here. TimeRadar's released model accepts 100 time steps and a variable number of channels. citeturn2view0

In [ ]:
from transformers import AutoModel
try:
    model=AutoModel.from_pretrained(str(MODEL_DIR),trust_remote_code=True,local_files_only=True)
except Exception as e:
    print('TimeRadar load failed with modern Transformers:',repr(e))
    print('This is a compatibility issue in the public checkpoint, not a MAINTAIN training failure.')
    raise
model.eval().to(DEVICE)
print(type(model).__name__)
print('parameters:',sum(p.numel() for p in model.parameters()))

## 4. Zero-shot TimeRadar smoke test on MAINTAIN AI data

Our existing risk tensor has 24 steps, while the released TimeRadar checkpoint requires 100. To avoid pretending that 24→100 interpolation is a real production preprocessing pipeline, this cell uses a **diagnostic compatibility test only**: each 24-step/60-channel sequence is linearly resampled to 100 steps.

A positive result here means the pretrained model can execute on our multivariate telemetry. It does **not** mean TimeRadar is already a calibrated 24h/48h/7d risk predictor.

In [ ]:
def resize_time(a,n=100):
    old=np.linspace(0,1,a.shape[1]); new=np.linspace(0,1,n)
    out=np.empty((a.shape[0],n,a.shape[2]),dtype=np.float32)
    for b in range(a.shape[0]):
        for c in range(a.shape[2]): out[b,:,c]=np.interp(new,old,a[b,:,c])
    return out

idx=te[:8]
xb=np.asarray(X[idx],dtype=np.float32)
xb=resize_time(xb,100)
# Normalize each window/channel for a scale-robust zero-shot diagnostic.
xb=(xb-xb.mean(1,keepdims=True))/(xb.std(1,keepdims=True)+1e-6)
with torch.no_grad():
    out=model(input_values=torch.from_numpy(xb).to(DEVICE))
print('input:',xb.shape)
print('anomaly_scores:',tuple(out.anomaly_scores.shape))
print('reconstructions:',tuple(out.reconstructions.shape))
scores=out.anomaly_scores.detach().float().cpu().numpy()
print('mean anomaly score per sample:',scores.mean(1))

## 5. Test the pretrained model on normal vs pre-event windows

This is still zero-shot. We only compare score distributions; no threshold is learned and no weights are updated.

In [ ]:
# Use the existing labels only for descriptive evaluation.
n=min(64,len(te)); idx=te[:n]; xb=resize_time(np.asarray(X[idx],dtype=np.float32),100); xb=(xb-xb.mean(1,keepdims=True))/(xb.std(1,keepdims=True)+1e-6)
with torch.no_grad(): s=model(input_values=torch.from_numpy(xb).to(DEVICE)).anomaly_scores.mean(1).detach().float().cpu().numpy()
label=Y[idx,0]
print('samples:',n,'positive 24h:',int(label.sum()))
print('score normal mean:',float(s[label==0].mean()) if (label==0).any() else None)
print('score positive mean:',float(s[label==1].mean()) if (label==1).any() else None)
try:
    from sklearn.metrics import roc_auc_score,average_precision_score
    if len(np.unique(label))>1: print('zero-shot anomaly ROC-AUC:',roc_auc_score(label,s),'PR-AUC:',average_precision_score(label,s))
except Exception as e: print('metric skipped:',e)

## 6. Chronos-2: pretrained forecasting candidate

Chronos-2 is a genuinely pretrained multivariate forecasting model. It can be loaded from Hugging Face with `Chronos2Pipeline.from_pretrained("amazon/chronos-2")`. citeturn1search7turn1search10

We keep this as a forecasting/representation experiment, not a failure-risk model. The next cell is isolated because TimeRadar pins an older Transformers version.

In [ ]:
# Optional separate environment note: run this only after the TimeRadar cells above succeed.
print('Chronos-2 is pretrained and supports multivariate forecasting, but it should be tested in a clean runtime/environment because TimeRadar pins transformers==4.40.1.')
print('Recommended clean-runtime install: pip install -U "chronos-forecasting>=2.0"')
print('Then: from chronos import Chronos2Pipeline; pipeline=Chronos2Pipeline.from_pretrained("amazon/chronos-2",device_map="cuda")')

## 7. Decision record

If TimeRadar executes successfully, we have a real pretrained anomaly backbone available without training. If Chronos-2 executes in a clean runtime, we have a real pretrained multivariate forecasting backbone. Neither is automatically a 24h/48h/7d failure classifier; that task-specific head is the part we can later adapt using MAINTAIN AI outcomes.

In [ ]:
report={
 'version':'maintain-ai-pretrained-audit-v1.5',
 'training_performed':False,
 'TimeRadar':{'pretrained_checkpoint':'included_in_public_repo','sequence_length_required':100,'channels':'variable','task':'zero-shot anomaly detection'},
 'Chronos-2':{'pretrained_checkpoint':'amazon/chronos-2','task':'zero-shot multivariate forecasting','failure_risk_head':False},
 'ChronosAD':{'status':'not treated as zero-training end-to-end model','reason':'official pipeline adds a custom temporal block and documents training/testing'},
 'MAINTAIN_input_shape':list(map(int,X.shape[1:])),
 'next_step':'benchmark pretrained anomaly/forecast signals before task-specific adaptation'
}
json.dump(report,open(OUT/'pretrained_audit.json','w'),indent=2); print(json.dumps(report,indent=2))